In [72]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


# 0. Read the data

In [73]:
from config import PER_ORGANISM_PER_CLASS_ALLELES
DATA_PATH = "/home/denklewer/Data/schreiber/"
# read total data

total_df =  pd.read_csv("{}schreiber_total_data.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))

per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_mt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    per_allele_wt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")
    
def defineOrganism(allele_name):
    for organism_to_use in PER_ORGANISM_PER_CLASS_ALLELES.keys():
        per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use]
        for class_to_use, alleles in per_class_alleles.items():
            if allele_name in alleles:
                return organism_to_use
    print("organism not  found")
    return None


def defineClass(allele_name):
    organism =  defineOrganism(allele_name)
    per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism]
    for class_to_use, alleles in per_class_alleles.items():
        if allele_name in alleles:
            return class_to_use
    print("not found")
    return None

    
ALLELES

['H-2Ld', 'H-2IEd']

In [74]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

In [75]:
METHOD = 'mhcflurry'

## 0.1. make result dicts

In [76]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.2 Read decoy data

In [77]:


per_allele_decoy = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    
    per_allele_decoy[allele_name] = decoy_df.copy()

# 1. Predict with mhcflurry

In [78]:
# importing the predict module
from mhcflurry import Class1PresentationPredictor
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

In [79]:
from mhcgnomes import parse

# different allele naming.
TARGET_ALLELES = dict()

## allele normalization now with mhcgnomes
for allele_name in ALLELES:
    result = parse(allele_name)
    """
    parse(
        raw_name,
        only_class1=True,
        required_result_types=[Allele, AlleleWithoutGene, Gene],
        preferred_result_types=[Allele],
        use_allele_aliases=True,
        infer_class2_pairing=False,
        collapse_singleton_haplotypes=True,
        collapse_singleton_serotypes=True,
        raise_on_error=False)
    """
    target_allele_name = result.to_string()
    TARGET_ALLELES[allele_name] = target_allele_name


TARGET_ALLELES

{'H-2Ld': 'H2-L*d', 'H-2IEd': 'H2-EA*d-EB*d'}

In [80]:
def filterByLength(per_allele_df, length_column='length'):
    #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
    for allele_name in per_allele_df.keys():
        
        initial_len = len(per_allele_df[allele_name])
        if defineClass(allele_name) == 'I':
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 15]
        else:
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 30]
        result_len = len(per_allele_df[allele_name])
        print(f"Filtered out {initial_len - result_len} samples")
        

    
print(f"\t mt")
filterByLength(per_allele_mt_df)
print(f"\t wt")
filterByLength(per_allele_wt_df)

	 mt
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples


In [273]:
def predict_with_mhcflurry(per_allele_df , per_allele_ref_scores=None):
    per_allele_predictions = dict()
    
    predictor = Class1PresentationPredictor.load()
    supported_alleles =  predictor.supported_alleles
    
    for allele_name in ALLELES:
        df =  per_allele_df[allele_name].copy()
        target_allele = TARGET_ALLELES[allele_name]
        print(f"---{target_allele}")
        if target_allele in supported_alleles:

            peptides = df[df['peptide'].notnull()]['peptide'].values
            # affinity	best_allele	affinity_percentile	processing_score	presentation_score	presentation_percentile
            prediction_df = predictor.predict(alleles=[target_allele], peptides=peptides, include_affinity_percentile=True)
            result_df = df.copy()
            result_df.loc[result_df['peptide'].notnull(), 'affinity'] = prediction_df['affinity']
            result_df.loc[result_df['peptide'].notnull(), 'percentile_rank_ba'] = prediction_df['affinity_percentile']
            result_df.loc[result_df['peptide'].notnull(), 'processing_score'] = prediction_df['processing_score']
            result_df.loc[result_df['peptide'].notnull(), 'score_el'] = prediction_df['presentation_score']
            result_df.loc[result_df['peptide'].notnull(), 'percentile_rank_el'] = prediction_df['presentation_percentile']
            
            
            # calculate percentile ranks if mhcflurry doesn't have such scores for allele
            allele_to_percent_rank_transform = predictor.affinity_predictor.allele_to_percent_rank_transform
            if target_allele not in allele_to_percent_rank_transform:
                if per_allele_ref_scores:
                    print("Calculate percentile ranks")
                    scores  = 1 - np.log(result_df['affinity'])/np.log(50000)
                    ref_scores = 1 - np.log(per_allele_ref_scores[allele_name]['affinity'])/np.log(50000)
                    ref_scores = ref_scores.sort_values()
                    pct_increment = 1./len(ref_scores)
                    percentile_ranks = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5*ref_scores.searchsorted(scores, side='right'))*pct_increment
                    percentile_ranks = 100 - percentile_ranks*100
                    result_df['percentile_rank_ba'] = percentile_ranks                
        else:
            #dummy prediction
            print("Method doesn't support {}".format(target_allele))
            result_df = df.copy()
            result_df['affinity'] = np.nan
            result_df['percentile_rank_ba'] = np.nan
            result_df['processing_score'] = np.nan
            result_df['score_el'] = np.nan
            result_df['percentile_rank_el'] = np.nan
        per_allele_predictions[allele_name] = result_df
        
    return per_allele_predictions
    

## create tempfiles for mhcnuggets



## 0.2 Read decoy data

In [84]:


per_allele_decoy = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    
    per_allele_decoy[allele_name] = decoy_df.copy()

In [85]:
%%time
print("decoy ")
filterByLength(per_allele_decoy)


decoy 
Filtered out 0 samples
Filtered out 0 samples
CPU times: user 28 ms, sys: 65 µs, total: 28.1 ms
Wall time: 26.8 ms


## First calculate scores for decoy

In [274]:
%%time
per_allele_decoy_predictions = predict_with_mhcflurry(per_allele_decoy)

  0%|          | 0/2 [00:00<?, ?it/s]

---H2-L*d
Predicting processing.


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automat

Predicting affinities.


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
100%|██████████| 1/1 [02:25<00:00, 145.38s/it]


---H2-EA*d-EB*d
Method doesn't support H2-EA*d-EB*d
CPU times: user 7min 41s, sys: 16.2 s, total: 7min 58s
Wall time: 5min 46s


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-L*d has no percentile rank information
  warnings.warn(msg)


In [275]:
per_allele_decoy_predictions[ALLELES[0]]

,peptide,length,affinity,percentile_rank_ba,processing_score,score_el,percentile_rank_el
0,NRTYSFQR,8,29723.133895,NaN,0.240352,0.007921,31.643179
1,AFYSGHGA,8,27024.125205,NaN,0.399518,0.015147,15.160951
2,AKKQSLLD,8,31746.997514,NaN,0.028266,0.003524,99.286603
3,RWLVVNVAHDLL,12,28184.850252,NaN,0.196215,0.007140,31.643179
4,ARRGDLDVLRSL,12,14599.653062,NaN,0.119942,0.010295,22.192310
...,...,...,...,...,...,...,...
199995,PVINDILTLALS,12,29996.893237,NaN,0.012550,0.003522,99.286603
199996,QIFTGNVPGAG,11,26134.821895,NaN,0.030820,0.004292,62.744674
199997,PRESHDKN,8,30758.143529,NaN,0.000332,0.003293,99.286603
199998,IHSGAKPYSC,10,18482.042755,NaN,0.026339,0.005905,46.224674


In [276]:
%%time
print(f"--mt")
per_allele_mt_predictions = predict_with_mhcflurry(per_allele_mt_df, per_allele_decoy_predictions)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions  

print(f"--wt")
per_allele_wt_predictions = predict_with_mhcflurry(per_allele_wt_df, per_allele_decoy_predictions)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions  


--mt


  0%|          | 0/1 [00:00<?, ?it/s]

---H2-L*d
Predicting processing.


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automat

Predicting affinities.


100%|██████████| 1/1 [01:37<00:00, 97.50s/it]
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-L*d has no percentile rank information
  warnings.warn(msg)


... (4 outputs left out)

Predicting affinities.


100%|██████████| 1/1 [01:57<00:00, 117.46s/it]

Calculate percentile ranks
---H2-EA*d-EB*d
Method doesn't support H2-EA*d-EB*d
CPU times: user 7min 23s, sys: 359 ms, total: 7min 24s
Wall time: 7min 23s



/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-L*d has no percentile rank information
  warnings.warn(msg)


In [277]:
per_method_per_allele_wt_predictions[METHOD][ALLELES[0]]

,peptide,length,allele,source_sequence_name,affinity,percentile_rank_ba,processing_score,score_el,percentile_rank_el
0,NCLPSLPS,8,H-2Ld,seq0,30392.479014,90.65675,0.000910,0.003338,99.286603
1,CLPSLPSI,8,H-2Ld,seq1,19667.399426,20.29775,0.031418,0.005661,46.224674
2,LPSLPSII,8,H-2Ld,seq2,9709.952154,5.13775,0.058369,0.012279,18.617989
3,PSLPSIIP,8,H-2Ld,seq3,29547.854525,83.54425,0.008946,0.003529,99.286603
4,SLPSIIPW,8,H-2Ld,seq4,27754.666252,66.92725,0.104440,0.005248,46.224674
5,LPSIIPWG,8,H-2Ld,seq5,28775.128978,76.54775,0.034114,0.003956,99.286603
6,PSIIPWGL,8,H-2Ld,seq6,22015.824427,28.25175,0.060974,0.005633,46.224674
7,SIIPWGLI,8,H-2Ld,seq7,25945.069949,51.34775,0.082587,0.005187,46.224674
8,ANCLPSLPS,9,H-2Ld,seq8,25574.611913,48.53025,0.005293,0.004006,62.744674
9,NCLPSLPSI,9,H-2Ld,seq9,4740.508083,1.95575,0.016398,0.021052,11.115788


# 2. Join predictions

In [278]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank'], 
    'mhcflurry':['source_sequence_name', 'affinity', 'percentile_rank_ba', 'processing_score', 'score_el', 'percentile_rank_el']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
    

# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [279]:
def joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print(f"method {method}")
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                
                

              

In [280]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

allele H-2Ld
method mhcflurry
allele H-2IEd
method mhcflurry


# 5. For now just write to files

In [281]:
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [282]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)

In [284]:
per_allele_result[ALLELES[0]]

,peptide_mt,length,allele,source_sequence_name,mhcflurry_affinity_mt,mhcflurry_percentile_rank_ba_mt,mhcflurry_processing_score_mt,mhcflurry_score_el_mt,mhcflurry_percentile_rank_el_mt,peptide_wt,mhcflurry_affinity_wt,mhcflurry_percentile_rank_ba_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcflurry_percentile_rank_el_wt
0,NCLPSLPF,8,H-2Ld,seq0,26212.381033,53.47475,0.001545,0.003860,99.286603,NCLPSLPS,30392.479014,90.65675,0.000910,0.003338,99.286603
1,CLPSLPFI,8,H-2Ld,seq1,23536.921670,35.41625,0.070986,0.005471,46.224674,CLPSLPSI,19667.399426,20.29775,0.031418,0.005661,46.224674
2,LPSLPFII,8,H-2Ld,seq2,11778.838561,7.00725,0.093166,0.011521,20.232255,LPSLPSII,9709.952154,5.13775,0.058369,0.012279,18.617989
3,PSLPFIIP,8,H-2Ld,seq3,29474.635340,82.90675,0.045090,0.004018,62.744674,PSLPSIIP,29547.854525,83.54425,0.008946,0.003529,99.286603
4,SLPFIIPW,8,H-2Ld,seq4,28884.552021,77.54425,0.083571,0.004692,62.744674,SLPSIIPW,27754.666252,66.92725,0.104440,0.005248,46.224674
5,LPFIIPWG,8,H-2Ld,seq5,26367.476293,54.76825,0.145780,0.006378,37.359049,LPSIIPWG,28775.128978,76.54775,0.034114,0.003956,99.286603
6,PFIIPWGL,8,H-2Ld,seq6,25435.882594,47.51425,0.145851,0.006605,37.359049,PSIIPWGL,22015.824427,28.25175,0.060974,0.005633,46.224674
7,FIIPWGLI,8,H-2Ld,seq7,26631.340917,56.99325,0.124353,0.005858,46.224674,SIIPWGLI,25945.069949,51.34775,0.082587,0.005187,46.224674
8,ANCLPSLPF,9,H-2Ld,seq8,12577.701110,7.83325,0.007284,0.008006,31.643179,ANCLPSLPS,25574.611913,48.53025,0.005293,0.004006,62.744674
9,NCLPSLPFI,9,H-2Ld,seq9,9126.465285,4.68425,0.017709,0.011305,20.232255,NCLPSLPSI,4740.508083,1.95575,0.016398,0.021052,11.115788
